# Complete Energy-Profile Comparison — LaTeX tables & boxplot

Same distance-to-real feature scorecard as `results_energy_profile_compare`, but
adds:
1. the **aggregated** scorecard as LaTeX,
2. one scorecard **per process** (separated) as LaTeX, and
3. a **boxplot across all processes** (per method), styled like the sMAE boxplot
   in `results_latex_table`.

Methods: Alpha, Combined-best, Budget (process types, best duration approach per
process), plus Schedule-direct and Profile-generator. Metric = relative
Wasserstein distance to real of each per-curve feature; **lower = closer to real**.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import os, glob, warnings
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import wasserstein_distance
from IPython.display import display, Markdown
import matplotlib.pyplot as plt, matplotlib.colors as mcolors
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

EXPERIMENT     = 961
CURVE_APPROACH = 'ml_exog_prev_activity'
DURATION_MODE  = 'best_by_wape'
PROCESS_TYPES  = {'Alpha': True, 
                  'Combined-best': True, 
                  'Budget': True}
EXTRA_METHODS  = {'Schedule-direct': True, 'Profile-generator': True}
DISABLED_MINERS = ['wip_aware', 'wip_branching_aware']
SELECTION_METRIC_BASES = ['control_flow_metrics_edge_f1_error',
                          'conformance_metrics_fitness_error',
                          'conformance_metrics_precision_error']
SAVE_LATEX = True

results_root = Path('..') / 'results'
runs = sorted([d for d in results_root.iterdir()
               if d.is_dir() and d.name.startswith(f'experiment_{EXPERIMENT}_')])
assert runs, f'No runs for experiment {EXPERIMENT}'
RUN = runs[-1]
print('Using run:', RUN.name)
SCHEDULE_SERIES = {'schedule': 'Schedule-direct', 'stochastic': 'Profile-generator'}
_curve_suffix = '' if CURVE_APPROACH in ('baseline', '', None) else f'_{CURVE_APPROACH}'

In [ ]:
# ── Per-curve features (adds 'median') ───────────────────────────────────────
def curve_features(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if v.size < 4: return None
    rng = np.nanmax(v) - np.nanmin(v)
    zero = np.mean((v - np.nanmin(v)) <= (0.02 * rng if rng > 0 else 1e-9))
    ac1 = pd.Series(v).autocorr(lag=1)
    return {'total': float(np.nansum(v)), 'peak': float(np.nanmax(v)),
            'mean': float(np.nanmean(v)), 'median': float(np.nanmedian(v)),
            'std': float(np.nanstd(v)), 'ac1': float(ac1) if np.isfinite(ac1) else np.nan,
            'roughness': float(np.nanmean(np.abs(np.diff(v)))), 'zero_frac': float(zero)}

FEATURES = ['total', 'peak', 'mean', 'median', 'std', 'ac1', 'roughness', 'zero_frac']
FEAT_LABEL = {'total':'Total','peak':'Peak','mean':'Mean','median':'Median','std':'Std',
              'ac1':'AC1','roughness':'Roughness','zero_frac':'Zero-frac'}

def features_long(df_curves, method_label):
    rows = []
    for (sen, cid), g in df_curves.groupby(['sensor', 'case_id']):
        f = curve_features(g.sort_values('t_minutes')['value'].to_numpy())
        if f:
            f.update(sensor=sen, case_id=cid, series=method_label); rows.append(f)
    return rows

In [ ]:
# ── Resolve process-type -> concrete simulation mode per process ─────────────
pe = pd.read_parquet(RUN / 'process_eval_results.parquet'); sp = 'test_'
def parse_mode(m):
    r = str(m)[len('petri_net_'):] if str(m).startswith('petri_net_') else str(m)
    if r.endswith('_ml_plus_global'): return r[:-len('_ml_plus_global')], 'ml_global'
    if r.endswith('_ml_plus_per_act'): return r[:-len('_ml_plus_per_act')], 'ml_local'
    return r, 'baseline'
pe['model'], pe['time_pred'] = zip(*pe['mode'].map(parse_mode))
pe = pe[~pe['model'].isin(DISABLED_MINERS)]
_sel_cols = [sp + b for b in SELECTION_METRIC_BASES if (sp + b) in pe.columns]
_cands = sorted(set(pe['model'].unique()) - {'alpha', 'budget'} - set(DISABLED_MINERS))
best_miner = {}
for proc, g in pe.groupby('process'):
    gc = g[g['model'].isin(_cands)]
    best_miner[proc] = gc.groupby('model')[_sel_cols].mean().mean(axis=1).idxmin() if not gc.empty else None
_TIME_SUFFIX = {'baseline':'', 'ml_global':'_ml_plus_global', 'ml_local':'_ml_plus_per_act'}
_wape_col = sp + 'duration_metrics_activity_duration_wape'
def model_for(proc, ptype):
    return {'Alpha':'alpha','Budget':'budget'}.get(ptype) or best_miner.get(proc)
def mode_for(proc, ptype):
    model = model_for(proc, ptype)
    if model is None: return None
    if DURATION_MODE in _TIME_SUFFIX: tp = DURATION_MODE
    else:
        sub = pe[(pe['process']==proc)&(pe['model']==model)]
        tp = sub.loc[sub[_wape_col].idxmin(),'time_pred'] if not sub.empty and _wape_col in sub else 'baseline'
    return f'petri_net_{model}{_TIME_SUFFIX[tp]}'

In [ ]:
# ── Build unified feature table (real + every method) ────────────────────────
active_ptypes = [t for t in ['Alpha','Combined-best','Budget'] if PROCESS_TYPES.get(t)]
active_extra  = [s for s,lbl in SCHEDULE_SERIES.items() if EXTRA_METHODS.get(lbl)]
METHOD_ORDER  = active_ptypes + [SCHEDULE_SERIES[s] for s in active_extra]

all_rows = []
for proc in sorted(pe['process'].unique()):
    for ptype in active_ptypes:
        mode = mode_for(proc, ptype)
        fp = RUN/'complete_curve_eval_results'/proc/(mode or '')/f'predicted_curves{_curve_suffix}.parquet'
        if not mode or not fp.exists(): continue
        d = pd.read_parquet(fp)
        recs = features_long(d[d['series']=='predicted'], ptype)
        for r in recs: r['process']=proc
        all_rows += recs
    sfp = RUN/'schedule_profile_eval_results'/proc/'predicted_curves.parquet'
    if not sfp.exists(): continue
    sdf = pd.read_parquet(sfp)
    for r in (rr for rr in features_long(sdf[sdf['series']=='real'], 'real')): r['process']=proc; all_rows.append(r)
    for s in active_extra:
        recs = features_long(sdf[sdf['series']==s], SCHEDULE_SERIES[s])
        for r in recs: r['process']=proc
        all_rows += recs
feat = pd.DataFrame(all_rows)

# restrict to sensors common to real + every method within each process
keep = []
for proc, g in feat.groupby('process'):
    sets = [set(g[g.series==m]['sensor'].unique()) for m in (['real']+METHOD_ORDER) if m in set(g.series)]
    common = set.intersection(*sets) if sets else set()
    keep.append(g[g['sensor'].isin(common)])
feat = pd.concat(keep, ignore_index=True)
print('feat rows:', len(feat), '| methods:', METHOD_ORDER)

In [ ]:
# ── Relative W1-to-real per (process, sensor, method, feature) — the raw grid ─
records = []
for (proc, sen), g in feat.groupby(['process', 'sensor']):
    for feature in FEATURES:
        r = g[g.series=='real'][feature].dropna().to_numpy()
        if len(r) < 2: continue
        scale = np.nanmean(np.abs(r)) + 1e-9
        # Degenerate-scale guard: for zero-inflated energy sensors the per-case
        # 'median' (and occasionally others) is ~0 for every real case, so
        # mean|real| collapses and the relative W1 explodes. Skip those cells
        # (they show '—') rather than emit meaningless ~1e9 values.
        if scale < 1e-6:
            continue
        for m in METHOD_ORDER:
            q = g[g.series==m][feature].dropna().to_numpy()
            if len(q) < 2: continue
            records.append({'process':proc,'sensor':sen,'method':m,'feature':feature,
                            'rel_w1': wasserstein_distance(r, q)/scale})
w1 = pd.DataFrame(records)
print('W1 grid rows:', len(w1))

def scorecard(df_w1):
    t = (df_w1.groupby(['method','feature'])['rel_w1'].median().unstack('feature')
             .reindex(index=METHOD_ORDER, columns=FEATURES))
    t.columns = [FEAT_LABEL[c] for c in t.columns]
    t.index.name = 'Method'
    return t

def style_score(tbl):
    return (tbl.style.format('{:.3f}', na_rep='—')
              .highlight_min(axis=0, props='font-weight:700;background-color:#d6ecff;')
              .set_caption('Relative W1 to real (median over process×sensor) — lower = closer to real'))

## 1 · Aggregated scorecard (all processes)

In [ ]:
score_all = scorecard(w1)
display(style_score(score_all))

## 2 · Per process (separated)

In [ ]:
per_process_scores = {}
for proc in sorted(w1['process'].unique()):
    s = scorecard(w1[w1['process']==proc])
    per_process_scores[proc] = s
    display(Markdown(f'### {proc}'))
    display(style_score(s))

## 3 · LaTeX — aggregated + one table per process

In [ ]:
def to_latex_score(tbl, caption, label):
    str_df = pd.DataFrame(index=tbl.index, columns=tbl.columns, dtype=object)
    for col in tbl.columns:
        vals = tbl[col].dropna(); best = vals.min() if not vals.empty else None
        for idx in tbl.index:
            v = tbl.loc[idx, col]
            if pd.isna(v): str_df.loc[idx, col] = '--'
            else:
                s = f'{v:.3f}'
                str_df.loc[idx, col] = (r'\textbf{'+s+'}') if (best is not None and abs(v-best)<1e-9) else s
    latex = str_df.to_latex(escape=False, column_format='l|'+'|'.join(['c']*len(tbl.columns)),
                            caption=caption, label=label, position='H')
    return latex.replace('_', r'\_')

out = RUN/'process_metrics_tables'
if SAVE_LATEX: out.mkdir(exist_ok=True)

tex_all = to_latex_score(score_all,
    caption=(f'Complete energy-profile comparison ({CURVE_APPROACH} curves), all processes. '
             r'Median over (process, sensor) of the relative Wasserstein distance to real per '
             r'feature; lower = closer to real. \textbf{Bold} = closest to real per feature.'),
    label=f'tab:energy_profile_all_{EXPERIMENT}')
if SAVE_LATEX: (out/'energy_profile_all.tex').write_text(tex_all)
print('% ===== ALL PROCESSES ====='); print(tex_all)

for proc, s in per_process_scores.items():
    tex = to_latex_score(s,
        caption=(f'Complete energy-profile comparison for {proc} ({CURVE_APPROACH} curves). '
                 r'Median over sensors of the relative Wasserstein distance to real per feature; '
                 r'lower = closer to real. \textbf{Bold} = closest to real per feature.'),
        label=f'tab:energy_profile_{proc}_{EXPERIMENT}')
    if SAVE_LATEX: (out/f'energy_profile_{proc}.tex').write_text(tex)
    print(f'\n% ===== {proc} ====='); print(tex)
if SAVE_LATEX: print('\nSaved per-process + aggregated .tex to', out)

## 4 · Boxplot across all processes

Composite realism distance per (process, sensor): the **median across features**
of the relative W1-to-real. One box per method over all (process, sensor) — same
horizontal style as the sMAE boxplot. Lower (left) = closer to real.

In [ ]:
# composite per (process, sensor, method) = median across features
comp = (w1.groupby(['process','sensor','method'])['rel_w1'].median().reset_index(name='val'))

plt.rcParams.update({'font.size': 12})
box_order = comp.groupby('method')['val'].median().sort_values(ascending=False).index.tolist()
data = [comp.loc[comp.method==m, 'val'].dropna().values for m in box_order]

fig, ax = plt.subplots(figsize=(8, 5))
bplot = ax.boxplot(data, tick_labels=box_order, showfliers=True, vert=False,
                   patch_artist=True, medianprops=dict(color='orange', linewidth=2))
blue = mcolors.to_rgba('blue', alpha=0.3)
for patch in bplot['boxes']:
    patch.set_facecolor(blue); patch.set_edgecolor('black')
all_vals = np.concatenate([d for d in data if len(d)])
ax.set_xlim(0, np.percentile(all_vals, 95) * 1.3)
ax.set_xlabel('Relative W1 to real (median across features)')
ax.set_ylabel('Method')
ax.legend(handles=[Line2D([0],[0],color='orange',lw=2,label='Median'),
                   Line2D([0],[0],marker='o',color='w',markeredgecolor='black',
                          markerfacecolor='none',markersize=6,label='Outlier')],
          loc='lower right')
fig.tight_layout()
Path('visuals').mkdir(exist_ok=True)
fig.savefig('visuals/energy_profile_W1_boxplot.pdf', dpi=300, bbox_inches='tight')
(RUN/'plots').mkdir(exist_ok=True)
fig.savefig(RUN/'plots'/'energy_profile_W1_boxplot.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: visuals/energy_profile_W1_boxplot.pdf and', RUN/'plots'/'energy_profile_W1_boxplot.png')

### Per-feature boxplots (the individual metrics)

In [ ]:
feats = FEATURES
ncol = 2; nrow = int(np.ceil(len(feats)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(12, 2.7*nrow))
axes = np.array(axes).reshape(-1)
for ax, feature in zip(axes, feats):
    sub = w1[w1.feature==feature]
    order = sub.groupby('method')['rel_w1'].median().sort_values(ascending=False).index.tolist()
    data = [sub.loc[sub.method==m,'rel_w1'].dropna().values for m in order]
    bp = ax.boxplot(data, tick_labels=order, showfliers=True, vert=False, patch_artist=True,
                    medianprops=dict(color='orange', linewidth=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor(mcolors.to_rgba('blue', alpha=0.3)); patch.set_edgecolor('black')
    av = np.concatenate([d for d in data if len(d)])
    if len(av): ax.set_xlim(0, np.percentile(av, 95)*1.3)
    ax.set_title(FEAT_LABEL[feature], fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=9)
for ax in axes[len(feats):]:
    ax.axis('off')
fig.suptitle('Relative W1 to real per feature — distribution over (process, sensor)',
             fontsize=13, fontweight='bold', y=1.005)
fig.tight_layout()
fig.savefig(RUN/'plots'/'energy_profile_W1_boxplot_per_feature.png', dpi=130, bbox_inches='tight')
plt.show()